<a href="https://colab.research.google.com/github/Emanuel600/Spotcar-Inversora/blob/dev/Planilha_de_C%C3%A1lculo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# imports
import numpy as np
# Dados básicos do sistema
## Entrada
Vin = 220  #[V RMS]
fin = 60   #[Hz]
## Saída
Io  = 200  #[A pico]
Vo  = 10   #[V RMS quadrado]
fs  = 40e3 #[Hz]
#===#
Vcc = Vin*np.sqrt(2) #[V] Tensão Vcc
a   = Vo/Vcc #[V/V] Assumindo meia ponte
Il  = 2*Io*a #[A] Corrente de carga espelhada para o primário
#===#
print(f"Relação de Transformação = {a} V/V")
print(f"Corrente no Primário = {Il} A (pico)")

Relação de Transformação = 0.03214121732666125 V/V
Corrente no Primário = 12.856486930664502 A (pico)


In [ ]:
# Capacitância Mínima para 1% de Variação em Vcc
dV = 1e-2 * Vcc   #[V]
C  = 2*Il/(fs*dV) #[F]
print(f"Capacitância Mínima para 1% = {1e6*C} uF")

Capacitância Mínima para 1% = 206.6115702479339 uF


In [ ]:
# Perda nas chaves - MOSFET (2SK3878)
tr  = 25e-9 #[s]
tf  = 20e-9 #[s]
Rds = 2     #[Ohm] - Corrigido para temp/corrente
#===#
Ps_Mos = fs*Vcc*Il*(tr+tf)   #[W] - Perda por Chaveamento
Pc_Mos = (Il/2)**2 * Rds     #[W] - Perda por Condução (Corrente RMS próxima a média - melhor caso)
P_Mos  = Ps_Mos + Pc_Mos     #[W] - Perdas Totais
#===#
print(f"Perda por chaveamento em ambas as chaves = {Ps_Mos} W")
print(f"Perda por condução em ambas as chaves = {Pc_Mos} W")
print(f"Perda total em ambas as chaves = {P_Mos} W")

Perda por chaveamento em ambas as chaves = 7.200000000000001 W
Perda por condução em ambas as chaves = 82.64462809917357 W
Perda total em ambas as chaves = 89.84462809917358 W


In [ ]:
# Perda nas chaves - IGBT (25T120)
tr  = 45e-9 #[s]
tf  = 75e-9 #[s]
Vce = 2.19  #[V] - @ Ic
#===#
Ps_Igbt = fs*Vcc*Il*(tr+tf)    #[W] - Perda por Chaveamento
Pc_Igbt = (Il/2) * Vce         #[W] - Perda por Condução (Corrente média)
P_Igbt  = Ps_Igbt + Pc_Igbt    #[W] - Perdas Totais
#===#
print(f"Perda por chaveamento em ambas as chaves = {Ps_Igbt} W")
print(f"Perda por condução em ambas as chaves = {Pc_Igbt} W")
print(f"Perda total em ambas as chaves = {P_Igbt} W")

Perda por chaveamento em ambas as chaves = 19.200000000000003 W
Perda por condução em ambas as chaves = 14.07785318907763 W
Perda total em ambas as chaves = 33.27785318907763 W


In [ ]:
# Cálculo de Dissipadores para as Chaves
Rt_Mos   = 0.833 #[°C/W]
Rt_Igbt  = 0.43  #[°C/W]
##
Tj_Mos   = 100 #[°C]
Tj_Igbt  = 120 #[°C]
##
Ta       = 40  #[°C]
dT_Mos   = Tj_Mos-Ta
dT_Igbt  = Tj_Igbt-Ta
##
Hrt_Mos  = 2*dT_Mos/P_Mos - Rt_Mos    # Fator de correção de 2 devido a
Hrt_Igbt = 2*dT_Igbt/P_Igbt - Rt_Igbt # potência sendo distribuida entre as chaves
##
print(f"Resistência térmica do heatsink do MOSFET = {Hrt_Mos} °C/W ({Hrt_Mos/(1.057*1.82)} °C/W/4\")") # Fator de correção para dT aplicado
print(f"Resistência térmica do heatsink do IGBT = {Hrt_Igbt} °C/W ({Hrt_Igbt/1.82} °C/W/4\")")

Resistência térmica do heatsink do MOSFET = 0.5026391198763702 °C/W (0.2612822522151487 °C/W/4")
Resistência térmica do heatsink do IGBT = 4.3780024601020475 °C/W (2.405495857198927 °C/W/4")


## Escolha de Dissipador
Tomando o catálogo da HS dissipadores como referência, temos que seria possível utilizar um dissipador similar ao 6835

## Adendo
Com relação à comparação entre MOSFETs e IGBTs, temos um MOSFET que pode ter perdas similares ao IGBT - o 2SK3878 com preço aproximado de 19,75 R$ por unidade

## Cálculo do Tranformador

Valores do tamanho do núcleo arredondados para baixo, mas com menos de 1mm de diferença

In [ ]:
Dw = 1  #[mm] - Diâmetro do fio (AWG 18)
Nw = 4  #[#]  - Número de fios
OD = 63 #[mm] - Diâmetro externo
ID = 33 #[mm] - Diâmetro interno
CH = 25 #[mm] - Largura do núcleo
###
WA = (ID/2)*CH       # Área da janela
Aw = np.pi*(Dw/2)**2 # Área de um fio
Ac = Nw*(1/a)*Aw     # Área total do cobre
###
Fill_Factor = Ac/WA
#
print(f"Fator de preenchimento do núcleo = {Fill_Factor*100}%")

Fator de preenchimento do núcleo = 23.695375670177953%
